# Laboratorio 2. El tablero no cuadra

Alguien dejó un pipeline que dice éxito. Finanzas dice que las ventas no cuadran con el archivo.

Su trabajo no es seguir una receta. Es dejar `oro_fact_ventas` usable y explicar qué apartaron.

Reglas del negocio, nada más:

- Una fila por `venta_id`.
- Solo entran las tiendas 1, 2 y 3.
- `monto` tiene que ser número. Si viene con coma, se interpreta. Si no se puede, se aparta.
- Una columna que no estaba en el contrato no entra al hecho.
- Lo apartado queda en `cuarentena_ventas`, con un motivo.
- Correr el notebook otra vez no duplica el hecho.

No editen el CSV. Si una prueba falla, el hallazgo es de ustedes.

In [0]:
# Rutas: los 4 CSV se suben SIN editar a un Volume de Unity Catalog
dbutils.widgets.text("ruta_base", "/Volumes/workspace/default/lab2")
base = dbutils.widgets.get("ruta_base").rstrip("/")
dbutils.widgets.text("ruta_csv", f"{base}/ventas_export.csv")
ruta = dbutils.widgets.get("ruta_csv")

def leer_csv(r):
    return spark.read.option("header", True).option("inferSchema", False).csv(r)

# Bronce: el archivo de ventas tal cual llega (todo como texto)
bronce = leer_csv(ruta)
bronce.createOrReplaceTempView("bronce_ventas")

# Contrato del negocio: solo estas tres tiendas entran al hecho
spark.sql("""
create or replace temp view dim_tienda as
select * from values
  (1, 'Centro'),
  (2, 'Escalon'),
  (3, 'Soyapango')
as t(tienda_id, nombre)
""")

# Archivos de apoyo (no se editan): maestro de tiendas, cierre de caja y ajustes tardios
leer_csv(f"{base}/tiendas.csv").selectExpr(
    "cast(tienda_id as int) as tienda_id", "nombre", "lower(trim(estado)) as estado"
).createOrReplaceTempView("tiendas_maestro")

leer_csv(f"{base}/cierre_caja.csv").selectExpr(
    "cast(fecha as date) as fecha", "cast(tienda_id as int) as tienda_id",
    "cast(total_caja as decimal(12,2)) as total_caja"
).createOrReplaceTempView("cierre_caja")

leer_csv(f"{base}/ajuste_tardio.csv").selectExpr(
    "cast(venta_id as int) as venta_id", "cast(monto_correcto as decimal(12,2)) as monto_correcto", "nota"
).createOrReplaceTempView("ajuste_tardio")

print("Filas en el archivo:", bronce.count())
print("Columnas:", bronce.columns)


Filas en el archivo: 59
Columnas: ['venta_id', 'fecha', 'tienda_id', 'producto_id', 'cantidad', 'monto', 'medio_pago', 'canal']


## El pipeline que heredaron

Esta celda es el flujo roto. Léanla, no la copien como entrega. Carga todo, no aparta nada y si la corren dos veces duplica.

In [0]:
spark.sql("""
create or replace temp view oro_roto as
select * from bronce_ventas
""")
display(spark.sql("select count(*) as filas_oro_roto from oro_roto"))

filas_oro_roto
59


## Su pipeline

Reemplacen esta celda. Tiene que crear `oro_fact_ventas` y `cuarentena_ventas`.

`oro_fact_ventas`: `venta_id`, `fecha`, `tienda_id`, `producto_id`, `cantidad`, `monto`, `medio_pago`.

`cuarentena_ventas`: las que no entran, con una columna `motivo`.

In [0]:
# Pipeline: crea oro_fact_ventas y cuarentena_ventas.
# Todo son vistas 'create or replace': correrlo otra vez REEMPLAZA, no agrega filas.
def correr_pipeline():
    # 1) Bronce con numero de fila (para distinguir copias identicas)
    spark.sql("""
    create or replace temp view bronce_numerado as
    select *, monotonically_increasing_id() as _fila
    from bronce_ventas
    """)

    # 2) Tipado y diagnostico de cada fila
    spark.sql(r"""
    create or replace temp view ventas_tipadas as
    select
      b.*,
      try_cast(trim(b.venta_id)    as int)  as venta_id_n,
      try_cast(trim(b.fecha)       as date) as fecha_n,
      try_cast(trim(b.tienda_id)   as int)  as tienda_id_n,
      try_cast(trim(b.producto_id) as int)  as producto_id_n,
      try_cast(trim(b.cantidad)    as int)  as cantidad_n,
      case
        when trim(b.monto) rlike '^-?[0-9]+([.][0-9]+)?$'
          then cast(trim(b.monto) as decimal(12,2))
        when trim(b.monto) rlike '^-?[0-9]+,[0-9]+$'
          then cast(replace(trim(b.monto), ',', '.') as decimal(12,2))
      end as monto_n,
      t.estado as estado_tienda,
      d.tienda_id is not null as tienda_en_contrato
    from bronce_numerado b
    left join tiendas_maestro t on try_cast(trim(b.tienda_id) as int) = t.tienda_id
    left join dim_tienda d      on try_cast(trim(b.tienda_id) as int) = d.tienda_id
    """)

    # 3) Primer motivo de rechazo (orden: tienda, monto, demas campos)
    spark.sql(r"""
    create or replace temp view ventas_evaluadas as
    select *,
      case
        when tienda_id_n is null                then 'tienda_no_numerica'
        when estado_tienda is null              then 'tienda_inexistente'
        when estado_tienda = 'cerrada'          then 'tienda_cerrada'
        when not tienda_en_contrato             then 'tienda_fuera_de_contrato'
        when monto is null or trim(monto) = ''  then 'monto_vacio'
        when trim(monto) rlike '^-?[0-9]{1,3}([.][0-9]{3})+,[0-9]+$'
                                                then 'monto_formato_miles_ambiguo'
        when monto_n is null                    then 'monto_no_numerico'
        when monto_n <= 0                       then 'monto_no_positivo'
        when venta_id_n is null                 then 'venta_id_invalido'
        when fecha_n is null                    then 'fecha_invalida'
        when producto_id_n is null              then 'producto_invalido'
        when cantidad_n is null or cantidad_n <= 0 then 'cantidad_invalida'
      end as motivo_regla
    from ventas_tipadas
    """)

    # 4) Candidatas: pasaron todas las reglas de fila
    spark.sql("""
    create or replace temp view candidatas as
    select *,
      size(collect_set(struct(fecha_n, tienda_id_n, producto_id_n, cantidad_n, monto_n, medio_pago))
        over (partition by venta_id_n)) as versiones,
      row_number() over (
        partition by venta_id_n, fecha_n, tienda_id_n, producto_id_n, cantidad_n, monto_n, medio_pago
        order by _fila) as copia
    from ventas_evaluadas
    where motivo_regla is null
    """)

    # 5) Ventas sin conflicto: una sola version (puede venir repetida)
    spark.sql("""
    create or replace temp view sin_conflicto as
    select * from candidatas where versiones = 1 and copia = 1
    """)

    # 6) Conflictos: misma venta_id con montos/datos distintos.
    #    Se resuelve con el cierre de caja: gana la version que hace cuadrar la caja del dia.
    spark.sql("""
    create or replace temp view evaluacion_conflictos as
    with otras as (
      select fecha_n, tienda_id_n, sum(monto_n) as suma_otras
      from sin_conflicto group by fecha_n, tienda_id_n
    ),
    versiones as (
      select c.*,
        abs(coalesce(o.suma_otras, 0) + c.monto_n - k.total_caja) < 0.005 as cuadra_caja
      from candidatas c
      left join otras o        on c.fecha_n = o.fecha_n and c.tienda_id_n = o.tienda_id_n
      left join cierre_caja k  on c.fecha_n = k.fecha   and c.tienda_id_n = k.tienda_id
      where c.versiones > 1 and c.copia = 1
    )
    select *,
      sum(case when cuadra_caja then 1 else 0 end) over (partition by venta_id_n) as versiones_que_cuadran
    from versiones
    """)

    # 7) HECHO ORO: contrato exacto de columnas (canal NO entra)
    spark.sql("""
    create or replace temp view oro_fact_ventas as
    select venta_id_n as venta_id, fecha_n as fecha, tienda_id_n as tienda_id,
           producto_id_n as producto_id, cantidad_n as cantidad, monto_n as monto, medio_pago
    from sin_conflicto
    union all
    select venta_id_n, fecha_n, tienda_id_n, producto_id_n, cantidad_n, monto_n, medio_pago
    from evaluacion_conflictos
    where versiones_que_cuadran = 1 and cuadra_caja
    """)

    # 8) CUARENTENA: fila original completa (incluye canal) + motivo
    spark.sql("""
    create or replace temp view cuarentena_ventas as
    select venta_id, fecha, tienda_id, producto_id, cantidad, monto, medio_pago, canal,
           motivo_regla as motivo
    from ventas_evaluadas where motivo_regla is not null
    union all
    select venta_id, fecha, tienda_id, producto_id, cantidad, monto, medio_pago, canal,
           'duplicado_exacto' as motivo
    from candidatas where copia > 1
    union all
    select venta_id, fecha, tienda_id, producto_id, cantidad, monto, medio_pago, canal,
           case when versiones_que_cuadran = 1 then 'duplicado_conflictivo_descartado_por_cierre_caja'
                else 'duplicado_conflictivo_sin_resolver' end as motivo
    from evaluacion_conflictos
    where not (versiones_que_cuadran = 1 and cuadra_caja)
    """)

    # 9) Ajustes tardios: se reportan, no se aplican al hecho
    spark.sql("""
    create or replace temp view ajustes_tardios_revision as
    select o.venta_id, o.fecha, o.tienda_id, o.monto as monto_en_hecho,
           a.monto_correcto, a.monto_correcto - o.monto as diferencia, a.nota
    from oro_fact_ventas o
    join ajuste_tardio a on o.venta_id = a.venta_id
    """)

    # 10) Conciliacion contra el cierre de caja (tiendas del contrato)
    spark.sql("""
    create or replace temp view conciliacion_caja as
    select k.fecha, k.tienda_id, k.total_caja,
           coalesce(sum(o.monto), 0) as total_hecho,
           coalesce(sum(o.monto), 0) - k.total_caja as diferencia
    from cierre_caja k
    join dim_tienda d on k.tienda_id = d.tienda_id
    left join oro_fact_ventas o on o.fecha = k.fecha and o.tienda_id = k.tienda_id
    group by k.fecha, k.tienda_id, k.total_caja
    """)

correr_pipeline()
print("Filas en bronce:     ", spark.table("bronce_ventas").count())
print("Filas en oro:        ", spark.table("oro_fact_ventas").count())
print("Filas en cuarentena: ", spark.table("cuarentena_ventas").count())


Filas en bronce:      59
Filas en oro:         25
Filas en cuarentena:  34


## Evidencia del pipeline

Cada celda muestra un resultado que respalda las decisiones. Estas salidas se capturan para el README del repositorio.

### 1. El hecho limpio

In [0]:
display(spark.sql("select * from oro_fact_ventas order by fecha, tienda_id, venta_id"))

venta_id,fecha,tienda_id,producto_id,cantidad,monto,medio_pago
1001,2026-10-01,1,501,2,18.50,efectivo
1002,2026-10-01,1,502,1,7.25,tarjeta
2001,2026-10-01,1,501,1,4.00,efectivo
3001,2026-10-01,1,510,1,6.75,efectivo
3002,2026-10-01,1,511,2,9.00,tarjeta
1004,2026-10-01,2,504,1,12.50,efectivo
3003,2026-10-01,2,512,1,8.00,efectivo
3004,2026-10-01,2,513,3,3.25,tarjeta
1005,2026-10-01,3,505,4,31.00,efectivo
3005,2026-10-01,3,514,1,14.00,transferencia


### 2. Qué se apartó y por qué

In [0]:
display(spark.sql("select motivo, count(*) as filas from cuarentena_ventas group by motivo order by filas desc"))
display(spark.sql("select * from cuarentena_ventas order by motivo, venta_id"))

motivo,filas
tienda_fuera_de_contrato,13
tienda_cerrada,5
duplicado_exacto,5
tienda_inexistente,4
monto_no_numerico,3
monto_vacio,1
monto_no_positivo,1
monto_formato_miles_ambiguo,1
duplicado_conflictivo_descartado_por_cierre_caja,1


venta_id,fecha,tienda_id,producto_id,cantidad,monto,medio_pago,canal,motivo
3050,2026-10-03,3,523,1,10.50,efectivo,tienda,duplicado_conflictivo_descartado_por_cierre_caja
1042,2026-10-02,1,501,1,18.50,efectivo,tienda,duplicado_exacto
2001,2026-10-01,1,501,1,4.00,efectivo,tienda,duplicado_exacto
3001,2026-10-01,1,510,1,6.75,efectivo,tienda,duplicado_exacto
3011,2026-10-02,2,516,2,19.90,efectivo,tienda,duplicado_exacto
3020,2026-10-03,1,520,2,21.00,efectivo,tienda,duplicado_exacto
3112,2026-10-01,2,536,1,"1.250,00",efectivo,tienda,monto_formato_miles_ambiguo
1202,2026-10-03,2,521,1,N/D,tarjeta,tienda,monto_no_numerico
3103,2026-10-04,3,527,1,N/D,efectivo,tienda,monto_no_numerico
3104,2026-10-01,1,528,1,pendiente,tarjeta,tienda,monto_no_numerico


### 3. Toda fila del archivo quedó en oro o en cuarentena

In [0]:
b = spark.table("bronce_ventas").count()
o = spark.table("oro_fact_ventas").count()
c = spark.table("cuarentena_ventas").count()
print(f"bronce={b}  oro={o}  cuarentena={c}  oro+cuarentena={o+c}")
print("Cuadra fila por fila" if o + c == b else "ATENCION: hay filas sin destino")

bronce=59  oro=25  cuarentena=34  oro+cuarentena=59
Cuadra fila por fila


### 4. Conciliación contra el cierre de caja de Finanzas (tiendas 1, 2 y 3)

In [0]:
display(spark.sql("select * from conciliacion_caja order by fecha, tienda_id"))
print("Dias-tienda con diferencia:", spark.sql("select count(*) from conciliacion_caja where diferencia <> 0").first()[0])

fecha,tienda_id,total_caja,total_hecho,diferencia
2026-10-01,1,45.50,45.50,0.00
2026-10-01,2,23.75,23.75,0.00
2026-10-01,3,45.00,45.00,0.00
2026-10-02,1,43.50,43.50,0.00
2026-10-02,2,26.00,26.00,0.00
2026-10-02,3,21.00,21.00,0.00
2026-10-03,1,21.00,21.00,0.00
2026-10-03,2,9.50,9.50,0.00
2026-10-03,3,27.75,27.75,0.00
2026-10-04,1,19.50,19.50,0.00


Dias-tienda con diferencia: 0


### 5. Ajustes tardíos: se reportan, no se aplican al hecho

In [0]:
display(spark.table("ajustes_tardios_revision"))

venta_id,fecha,tienda_id,monto_en_hecho,monto_correcto,diferencia,nota
1007,2026-10-02,1,15.00,12.00,-3.00,el cajero corrigio despues del cierre
1401,2026-10-04,1,5.50,5.00,-0.50,vuelto mal registrado
3010,2026-10-02,1,10.00,9.00,-1.00,descuento aplicado fuera de caja
3025,2026-10-04,1,14.00,14.50,0.50,recuento de estante


### 6. Idempotencia: correr el pipeline otra vez no hace crecer el hecho

In [0]:
antes = spark.table("oro_fact_ventas").count()
correr_pipeline()
despues = spark.table("oro_fact_ventas").count()
print(f"Antes: {antes}  Después de correrlo otra vez: {despues}")
print("No se duplicó" if antes == despues else "ATENCION: el hecho creció")

Antes: 25  Después de correrlo otra vez: 25
No se duplicó


## Pruebas

No dicen cómo arreglarlo. Dicen si ya quedó. Las seis tienen que decir PASÓ.

In [0]:
oro = spark.table("oro_fact_ventas")
cuarentena = spark.table("cuarentena_ventas")
bronce = spark.table("bronce_ventas")

def prueba(nombre, ok):
    print(("PASÓ  " if ok else "FALLÓ ") + nombre)
    return ok

duplicados = spark.sql("""
select count(*) c from (
  select venta_id from oro_fact_ventas
  group by venta_id having count(*) > 1
)""").first()[0]

tiendas_malas = spark.sql("""
select count(*) c from oro_fact_ventas
where try_cast(tienda_id as int) not in (1, 2, 3)
""").first()[0]

montos_malos = spark.sql("""
select count(*) c from oro_fact_ventas
where try_cast(monto as double) is null
""").first()[0]

columnas_de_mas = [c for c in oro.columns if c.lower() == "canal"]
sin_motivo = "motivo" not in [c.lower() for c in cuarentena.columns]
oro_inflado = oro.count() >= bronce.count()
cuarentena_vacia = cuarentena.count() == 0

ok = []
ok.append(prueba("una fila por venta_id", duplicados == 0))
ok.append(prueba("solo tiendas 1, 2 y 3", tiendas_malas == 0))
ok.append(prueba("monto numérico en el hecho", montos_malos == 0))
ok.append(prueba("canal no entra al hecho", len(columnas_de_mas) == 0))
ok.append(prueba("cuarentena con motivo y con filas", (not sin_motivo) and (not cuarentena_vacia)))
ok.append(prueba("oro menor que el archivo", not oro_inflado))
print("-")
print("LISTO" if all(ok) else "SIGAN. Hay pruebas en rojo.")

PASÓ  una fila por venta_id
PASÓ  solo tiendas 1, 2 y 3
PASÓ  monto numérico en el hecho
PASÓ  canal no entra al hecho
PASÓ  cuarentena con motivo y con filas
PASÓ  oro menor que el archivo
-
LISTO


## Desafío o reto

Cuando las seis digan PASÓ, respondan en esta celda, sin mirar la guía:

1. ¿Qué pasó con la misma venta llegada dos veces?
2. ¿Qué hicieron con un monto que venía en texto?
3. ¿Por qué una tienda se quedó fuera?
4. ¿Dónde quedó la columna que no estaba en el contrato?
5. Si corren el notebook otra vez, ¿el hecho crece?

El que solo pega la consulta y no puede explicar una de estas, no termino el laboratorio

## Respuestas del desafío

**1. ¿Qué pasó con la misma venta llegada dos veces?**
Hubo dos casos. Cuando la venta llegó repetida idéntica (2001, 3001, 1042, 3011 y 3020), dejé una sola copia en el hecho y la copia sobrante quedó en cuarentena como `duplicado_exacto`. La venta 3050 llegó dos veces con montos distintos (10.00 y 10.50), así que no era una copia: era un conflicto. Lo resolví con el cierre de caja de Finanzas: la tienda 3 cerró el 3 de octubre con 27.75, y solo la versión de 10.00 hace cuadrar ese total. Esa entró al hecho y la de 10.50 quedó en cuarentena como `duplicado_conflictivo_descartado_por_cierre_caja`.

**2. ¿Qué hicieron con un monto que venía en texto?**
Si traía coma decimal (`"12,50"`, `"3,25"`), la reemplacé por punto y la convertí a número. Si no se podía interpretar (vacío, `N/D`, `pendiente`), la aparté con su motivo. También aparté `"1.250,00"`: se podría leer como 1250, pero el cierre de caja de la tienda 2 ese día fue 23.75, así que esa venta nunca entró a caja y la dejé para revisión como `monto_formato_miles_ambiguo`. El monto negativo (-4.00) quedó como `monto_no_positivo`.

**3. ¿Por qué una tienda se quedó fuera?**
Porque el contrato solo admite las tiendas 1, 2 y 3. Separé el motivo para que Finanzas sepa qué hacer con cada caso: las tiendas 99, 88 y 77 no existen en el maestro (`tienda_inexistente`); la 4 y la 7 están cerradas (`tienda_cerrada`); y la 5 y la 6 están abiertas y tienen cierre de caja, pero no están en el contrato (`tienda_fuera_de_contrato`). Esto último es un hallazgo: si el negocio quiere reportar esas tiendas, hay que ampliar el contrato.

**4. ¿Dónde quedó la columna que no estaba en el contrato?**
`canal` sigue en bronce, que guarda el archivo tal cual llegó, y en la cuarentena, para revisar el registro completo. No entra a `oro_fact_ventas`, porque el hecho solo tiene las 7 columnas del contrato.

**5. Si corren el notebook otra vez, ¿el hecho crece?**
No. Todas las tablas se crean con `create or replace`, que reemplaza el resultado en lugar de agregarle filas, y la deduplicación se hace dentro del pipeline. La celda de idempotencia lo comprueba: el hecho tiene 25 filas antes y después de volver a correrlo.
